## 🎯 Learning Objectives
* Design and implement a structured prompt evaluation harness.
* Define and apply various evaluation criteria for LLM responses, including keyword matching, regex, and custom logic.
* Automate the process of testing prompts against a suite of test cases.
* Generate actionable reports to inform prompt iteration and improvement.


## Exercise: Building a Prompt Evaluation Harness

**Lesson ID:** LLM03-L11
**Lesson Title:** Building a prompt eval harness

### Task Description

In this exercise, you will build a robust and extensible prompt evaluation harness. This harness will allow you to systematically test different prompts against a predefined set of test cases, evaluate the LLM's responses based on specific criteria, and report the results. This is a critical component for iterating on prompts and ensuring their quality and reliability in production.

### Requirements

Your prompt evaluation harness should meet the following requirements:

1.  **Test Case Definition:** Allow for the definition of test cases, where each test case includes:
    *   An `input` (the user query or context for the LLM).
    *   An `expected_output_criteria` (a dictionary or list of criteria to evaluate the LLM's response against).
    *   An optional `prompt_template_name` to specify which prompt template to use if multiple are being tested.

2.  **LLM Interaction:** Integrate with a mock LLM API (provided in the setup code) to simulate calling a real LLM. Your harness should be designed to easily swap out the mock LLM for a real one (e.g., OpenAI, Anthropic, custom local model) in a production setting.

3.  **Evaluation Strategies:** Support multiple evaluation strategies for `expected_output_criteria`:
    *   **Keyword Presence:** Check if specific keywords or phrases are present in the LLM's response.
    *   **Regular Expression Matching:** Check if the LLM's response matches a given regular expression pattern.
    *   **Custom Function:** Allow for a custom Python function to be passed, which takes the LLM response and returns a boolean (pass/fail) or a score.
    *   **Semantic Similarity (Optional but Recommended for 2026):** If you're feeling ambitious, consider how you might integrate a semantic similarity check (e.g., using an embedding model) to compare the LLM's response with an expected answer.

4.  **Reporting:** Generate a clear and concise report summarizing the evaluation results. This report should include:
    *   Total number of test cases.
    *   Number of passed and failed test cases.
    *   Overall pass rate (accuracy).
    *   Detailed results for each test case, including input, LLM response, expected criteria, and whether it passed or failed, along with any specific failure reasons.

5.  **Extensibility:** The design should be modular, making it easy to add new evaluation criteria or integrate with different LLM providers in the future.

### Evaluation Criteria

Your solution will be evaluated based on:

*   **Correctness:** Does the harness correctly evaluate responses based on the defined criteria?
*   **Completeness:** Does it meet all the specified requirements?
*   **Readability and Maintainability:** Is the code clean, well-structured, and easy to understand?
*   **Modularity:** Is the design extensible for future additions?
*   **Robustness:** Does it handle edge cases gracefully (e.g., empty responses, criteria not met)?
*   **Clarity of Reporting:** Is the generated report informative and easy to interpret?

Good luck! This exercise will solidify your understanding of systematic prompt engineering.


In [ ]:
import json
import time
import re
from typing import List, Dict, Any, Callable, Union, Optional

# --- Mock LLM API (Simulates a real LLM call) ---
class MockLLM:
    """A mock LLM class to simulate API calls and responses."""
    def __init__(self, responses: Dict[str, str] = None, default_response: str = "I don't know."):
        self.responses = responses if responses is not None else {}
        self.default_response = default_response

    def generate(self, prompt: str, model: str = "gpt-4o-2026-01-01", temperature: float = 0.7) -> str:
        """Simulates an LLM generating a response based on the prompt."""
        print(f"[MockLLM] Generating response for prompt: {prompt[:50]}...")
        time.sleep(0.1) # Simulate network latency
        # A very basic way to get a 'canned' response based on prompt content
        for key, value in self.responses.items():
            if key in prompt:
                return value
        return self.default_response

# --- Example Prompt Templates (for demonstration) ---
PROMPT_TEMPLATES = {
    "summarizer": "Summarize the following text concisely: {text}",
    "qa_extractor": "Extract the key information and answer the question: {question}\nContext: {context}",
    "sentiment_analyzer": "Analyze the sentiment of the following text (positive, negative, neutral): {text}"
}

# --- Helper Evaluation Functions (for custom criteria) ---
def check_contains_keywords(response: str, keywords: List[str], case_sensitive: bool = False) -> bool:
    """Checks if all specified keywords are present in the response."""
    response_to_check = response if case_sensitive else response.lower()
    return all(k in response_to_check for k in (keywords if case_sensitive else [k.lower() for k in keywords]))

def check_regex_match(response: str, pattern: str) -> bool:
    """Checks if the response matches the given regex pattern."""
    return bool(re.search(pattern, response))

def check_sentiment_positive(response: str) -> bool:
    """Custom function to check if sentiment is positive."""
    return "positive" in response.lower()

# --- Mock Data for LLM Responses ---
mock_llm_responses = {
    "summarize the following text": "The quick brown fox jumps over the lazy dog. This is a classic English phrase often used for testing fonts and typewriters. It contains all letters of the alphabet.",
    "extract the key information": "The answer is: The quick brown fox jumps over the lazy dog. It's a pangram.",
    "analyze the sentiment": "The sentiment is positive. It's a fun phrase.",
    "What is the capital of France": "The capital of France is Paris.",
    "Who is the current president of the USA": "The current president of the USA is Joe Biden."
}
mock_llm = MockLLM(responses=mock_llm_responses, default_response="I cannot provide that information.")

# --- Example Test Cases ---
# Each test case includes input, expected criteria, and an optional prompt template name.
TEST_CASES = [
    {
        "id": "summary_test_1",
        "input": {"text": "The quick brown fox jumps over the lazy dog. This is a classic English phrase often used for testing fonts and typewriters. It contains all letters of the alphabet."},
        "prompt_template_name": "summarizer",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["fox", "dog", "alphabet"]},
            {"type": "regex_match", "value": r"fox.*dog"}
        ]
    },
    {
        "id": "qa_test_1",
        "input": {"question": "What is the capital of France?", "context": "France is a country in Western Europe. Its capital is Paris."},
        "prompt_template_name": "qa_extractor",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["Paris"]},
            {"type": "custom_function", "function": lambda resp: "capital" in resp.lower() and "paris" in resp.lower(), "description": "Response mentions capital and Paris"}
        ]
    },
    {
        "id": "sentiment_test_1",
        "input": {"text": "I absolutely love this new feature! It's fantastic."},
        "prompt_template_name": "sentiment_analyzer",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["positive"]},
            {"type": "custom_function", "function": check_sentiment_positive, "description": "Sentiment is positive"}
        ]
    },
    {
        "id": "qa_test_2_fail",
        "input": {"question": "Who is the current president of the USA?", "context": "The USA has a democratic government."},
        "prompt_template_name": "qa_extractor",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["Donald Trump"]}
        ] # This should fail with mock LLM
    },
    {
        "id": "summary_test_2_short",
        "input": {"text": "AI is transforming industries."},
        "prompt_template_name": "summarizer",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["AI", "industries"]}
        ]
    }
]

print("Setup complete. Mock LLM, prompt templates, and test cases are ready.")


### Your Implementation

Now it's your turn to build the `PromptEvaluator` class and the evaluation logic. Use the `MockLLM`, `PROMPT_TEMPLATES`, and `TEST_CASES` defined in the setup cell.

Your solution should include:

1.  A class `PromptEvaluator` that can take an LLM instance and prompt templates.
2.  A method to add test cases.
3.  A method to run the evaluation across all added test cases.
4.  A method to generate and print a comprehensive report.
5.  Implement the logic for `keyword_presence`, `regex_match`, and `custom_function` evaluation types.

Feel free to add any additional helper methods or classes you deem necessary for a robust solution. Remember to consider extensibility and clear reporting.


In [ ]:
import json
import time
import re
from typing import List, Dict, Any, Callable, Union, Optional

# Re-define MockLLM and helper functions for self-contained solution
# In a real notebook, these would be imported or defined once at the top.

class MockLLM:
    """A mock LLM class to simulate API calls and responses."""
    def __init__(self, responses: Dict[str, str] = None, default_response: str = "I don't know."):
        self.responses = responses if responses is not None else {}
        self.default_response = default_response

    def generate(self, prompt: str, model: str = "gpt-4o-2026-01-01", temperature: float = 0.7) -> str:
        """Simulates an LLM generating a response based on the prompt."""
        # print(f"[MockLLM] Generating response for prompt: {prompt[:50]}...") # Uncomment for verbose output
        time.sleep(0.05) # Simulate network latency, slightly faster for solution
        for key, value in self.responses.items():
            if key.lower() in prompt.lower(): # Case-insensitive matching for mock responses
                return value
        return self.default_response

PROMPT_TEMPLATES = {
    "summarizer": "Summarize the following text concisely: {text}",
    "qa_extractor": "Extract the key information and answer the question: {question}\nContext: {context}",
    "sentiment_analyzer": "Analyze the sentiment of the following text (positive, negative, neutral): {text}"
}

def check_contains_keywords(response: str, keywords: List[str], case_sensitive: bool = False) -> bool:
    """Checks if all specified keywords are present in the response."""
    response_to_check = response if case_sensitive else response.lower()
    return all(k in response_to_check for k in (keywords if case_sensitive else [k.lower() for k in keywords]))

def check_regex_match(response: str, pattern: str) -> bool:
    """Checks if the response matches the given regex pattern."""
    return bool(re.search(pattern, response))

def check_sentiment_positive(response: str) -> bool:
    """Custom function to check if sentiment is positive."""
    return "positive" in response.lower()

mock_llm_responses = {
    "summarize the following text": "The quick brown fox jumps over the lazy dog. This is a classic English phrase often used for testing fonts and typewriters. It contains all letters of the alphabet.",
    "extract the key information": "The answer is: The capital of France is Paris.",
    "analyze the sentiment": "The sentiment is positive. It's a fun phrase.",
    "What is the capital of France": "The capital of France is Paris.",
    "Who is the current president of the USA": "The current president of the USA is Joe Biden."
}
mock_llm = MockLLM(responses=mock_llm_responses, default_response="I cannot provide that information.")

TEST_CASES = [
    {
        "id": "summary_test_1",
        "input": {"text": "The quick brown fox jumps over the lazy dog. This is a classic English phrase often used for testing fonts and typewriters. It contains all letters of the alphabet."},
        "prompt_template_name": "summarizer",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["fox", "dog", "alphabet"]},
            {"type": "regex_match", "value": r"fox.*dog"}
        ]
    },
    {
        "id": "qa_test_1",
        "input": {"question": "What is the capital of France?", "context": "France is a country in Western Europe. Its capital is Paris."},
        "prompt_template_name": "qa_extractor",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["Paris"]},
            {"type": "custom_function", "function": lambda resp: "capital" in resp.lower() and "paris" in resp.lower(), "description": "Response mentions capital and Paris"}
        ]
    },
    {
        "id": "sentiment_test_1",
        "input": {"text": "I absolutely love this new feature! It's fantastic."},
        "prompt_template_name": "sentiment_analyzer",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["positive"]},
            {"type": "custom_function", "function": check_sentiment_positive, "description": "Sentiment is positive"}
        ]
    },
    {
        "id": "qa_test_2_fail",
        "input": {"question": "Who is the current president of the USA?", "context": "The USA has a democratic government."},
        "prompt_template_name": "qa_extractor",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["Donald Trump"]}
        ] # This should fail with mock LLM
    },
    {
        "id": "summary_test_2_short",
        "input": {"text": "AI is transforming industries."},
        "prompt_template_name": "summarizer",
        "expected_output_criteria": [
            {"type": "keyword_presence", "value": ["AI", "industries"]}
        ]
    }
]


class PromptEvaluator:
    """A class to systematically evaluate LLM prompts against defined test cases."""

    def __init__(self, llm_client: Any, prompt_templates: Dict[str, str]):
        """Initializes the PromptEvaluator.

        Args:
            llm_client: An instance of an LLM client (e.g., MockLLM, OpenAI client).
            prompt_templates: A dictionary of named prompt templates.
        """
        self.llm_client = llm_client
        self.prompt_templates = prompt_templates
        self.test_cases: List[Dict[str, Any]] = []
        self.results: List[Dict[str, Any]] = []

    def add_test_case(self, test_case: Dict[str, Any]):
        """Adds a single test case to the evaluator."""
        self.test_cases.append(test_case)

    def add_test_cases(self, test_cases: List[Dict[str, Any]]):
        """Adds multiple test cases to the evaluator."""
        self.test_cases.extend(test_cases)

    def _format_prompt(self, template_name: str, input_data: Dict[str, Any]) -> str:
        """Formats a prompt using the specified template and input data."""
        template = self.prompt_templates.get(template_name)
        if not template:
            raise ValueError(f"Prompt template '{template_name}' not found.")
        try:
            return template.format(**input_data)
        except KeyError as e:
            raise ValueError(f"Missing key in input data for template '{template_name}': {e}")

    def _evaluate_response(self, response: str, criteria: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Evaluates an LLM response against a list of criteria.

        Args:
            response: The LLM's generated response string.
            criteria: A list of dictionaries, each defining an evaluation criterion.

        Returns:
            A dictionary containing 'passed' (bool) and 'failure_reasons' (list of strings).
        """
        all_passed = True
        failure_reasons = []

        if not response:
            all_passed = False
            failure_reasons.append("LLM returned an empty response.")
            return {"passed": all_passed, "failure_reasons": failure_reasons}

        for criterion in criteria:
            crit_type = criterion.get("type")
            crit_value = criterion.get("value")
            crit_description = criterion.get("description", f"Criterion type: {crit_type}")
            criterion_passed = False

            try:
                if crit_type == "keyword_presence":
                    if isinstance(crit_value, list):
                        criterion_passed = check_contains_keywords(response, crit_value)
                    else:
                        raise ValueError("Keyword presence 'value' must be a list of strings.")
                elif crit_type == "regex_match":
                    if isinstance(crit_value, str):
                        criterion_passed = check_regex_match(response, crit_value)
                    else:
                        raise ValueError("Regex match 'value' must be a string pattern.")
                elif crit_type == "custom_function":
                    custom_func = criterion.get("function")
                    if callable(custom_func):
                        criterion_passed = custom_func(response)
                    else:
                        raise ValueError("Custom function 'function' must be a callable.")
                # Add more evaluation types here (e.g., semantic_similarity)
                # elif crit_type == "semantic_similarity":
                #     expected_text = crit_value["expected_text"]
                #     min_score = crit_value["min_score"]
                #     criterion_passed = self._check_semantic_similarity(response, expected_text, min_score)
                else:
                    raise ValueError(f"Unknown evaluation criterion type: {crit_type}")

            except Exception as e:
                criterion_passed = False
                failure_reasons.append(f"Error evaluating '{crit_description}': {e}")

            if not criterion_passed:
                all_passed = False
                failure_reasons.append(f"Failed criterion: {crit_description}")

        return {"passed": all_passed, "failure_reasons": failure_reasons}

    def run_evaluation(self):
        """Runs the evaluation for all added test cases."""
        self.results = [] # Clear previous results
        print(f"\n--- Starting Prompt Evaluation for {len(self.test_cases)} test cases ---")
        for i, test_case in enumerate(self.test_cases):
            test_id = test_case.get("id", f"test_case_{i+1}")
            prompt_template_name = test_case.get("prompt_template_name")
            input_data = test_case.get("input")
            expected_criteria = test_case.get("expected_output_criteria", [])

            print(f"Evaluating Test Case: {test_id}...")

            if not prompt_template_name or not input_data:
                self.results.append({
                    "id": test_id,
                    "status": "SKIPPED",
                    "reason": "Missing prompt_template_name or input data."
                })
                continue

            try:
                # 1. Format the prompt
                formatted_prompt = self._format_prompt(prompt_template_name, input_data)

                # 2. Call the LLM
                llm_response = self.llm_client.generate(formatted_prompt)

                # 3. Evaluate the response
                evaluation_result = self._evaluate_response(llm_response, expected_criteria)

                self.results.append({
                    "id": test_id,
                    "prompt_template": prompt_template_name,
                    "input": input_data,
                    "formatted_prompt": formatted_prompt,
                    "llm_response": llm_response,
                    "expected_criteria": expected_criteria,
                    "passed": evaluation_result["passed"],
                    "failure_reasons": evaluation_result["failure_reasons"]
                })

            except Exception as e:
                self.results.append({
                    "id": test_id,
                    "prompt_template": prompt_template_name,
                    "input": input_data,
                    "status": "ERROR",
                    "error": str(e)
                })
        print("--- Evaluation Complete ---")

    def generate_report(self):
        """Generates and prints a summary report of the evaluation results."""
        if not self.results:
            print("No evaluation results to report. Run `run_evaluation()` first.")
            return

        total_cases = len(self.results)
        passed_cases = sum(1 for r in self.results if r.get("passed", False))
        failed_cases = total_cases - passed_cases
        pass_rate = (passed_cases / total_cases) * 100 if total_cases > 0 else 0

        print("\n" + "="*50)
        print("Prompt Evaluation Report")
        print("="*50)
        print(f"Total Test Cases: {total_cases}")
        print(f"Passed: {passed_cases}")
        print(f"Failed: {failed_cases}")
        print(f"Pass Rate: {pass_rate:.2f}%")
        print("="*50)

        if failed_cases > 0:
            print("\nDetailed Failures:")
            for result in self.results:
                if not result.get("passed", False):
                    print(f"  Test Case ID: {result['id']}")
                    print(f"    Prompt Template: {result.get('prompt_template', 'N/A')}")
                    print(f"    Input: {json.dumps(result.get('input'), indent=2)}")
                    print(f"    LLM Response: {result.get('llm_response', 'N/A')[:200]}...")
                    print(f"    Failure Reasons:")
                    for reason in result.get('failure_reasons', []):
                        print(f"      - {reason}")
                    print("    ---")
        else:
            print("\nAll test cases passed successfully!")
        print("="*50)

# --- Instantiate and Run the Evaluator ---

# 1. Create an instance of the PromptEvaluator
evaluator = PromptEvaluator(llm_client=mock_llm, prompt_templates=PROMPT_TEMPLATES)

# 2. Add the test cases
evaluator.add_test_cases(TEST_CASES)

# 3. Run the evaluation
evaluator.run_evaluation()

# 4. Generate and print the report
evaluator.generate_report()

# Example of adding a new test case and re-running
print("\nAdding a new test case and re-running evaluation...")
new_test_case = {
    "id": "new_qa_test_success",
    "input": {"question": "What is the largest ocean?", "context": "The Pacific Ocean is the largest and deepest of Earth's five oceanic divisions."},
    "prompt_template_name": "qa_extractor",
    "expected_output_criteria": [
        {"type": "keyword_presence", "value": ["Pacific Ocean"]}
    ]
}
evaluator.add_test_case(new_test_case)

mock_llm.responses["What is the largest ocean"] = "The largest ocean is the Pacific Ocean."

evaluator.run_evaluation()
evaluator.generate_report()
